# LibFlow – Hybrid SQL and NoSQL DBMS Project
**Library Management System | PostgreSQL + MongoDB + Python**

This notebook demonstrates a hybrid database design for LibFlow. PostgreSQL stores structured operational records (students, books, issues, fines, payments). MongoDB stores flexible nested book metadata (editions, reviews, keywords and related topics).

**Important:** Run the SQL schema and sample-data scripts in PostgreSQL first. Add your connection strings as environment variables or Colab Secrets. Do not publish passwords or connection strings in screenshots.

## 1. Objectives
1. Design relational tables with primary keys, foreign keys and integrity constraints.
2. Store flexible, nested metadata in MongoDB.
3. Connect Python to both databases.
4. Synchronize selected MongoDB fields into a PostgreSQL index using the stable `book_id`.
5. Demonstrate CRUD, JOIN, nested SQL query, MongoDB nested query and a transaction.
6. Explain why a hybrid design is useful and when SQL-only would be sufficient.

## 2. Install Python packages

In [ ]:
!pip -q install psycopg2-binary pymongo dnspython pandas

## 3. Configure database connections
Create a PostgreSQL database (for example `libflow_db`) and run `LibFlow_Schema.sql`, followed by `LibFlow_Sample_Data.sql`.

Set these environment variables in Colab Secrets or your local environment:
- `PGHOST`, `PGPORT`, `PGDATABASE`, `PGUSER`, `PGPASSWORD`
- `MONGO_URI`, `MONGO_DATABASE` (example database name: `libflow`)

In [ ]:
import os
import pandas as pd
import psycopg2
from psycopg2.extras import execute_values
from pymongo import MongoClient
from datetime import date, timedelta

pg_conn = psycopg2.connect(
    host=os.environ["PGHOST"],
    port=os.environ.get("PGPORT", "5432"),
    dbname=os.environ["PGDATABASE"],
    user=os.environ["PGUSER"],
    password=os.environ["PGPASSWORD"]
)
pg_conn.autocommit = False

mongo_client = MongoClient(os.environ["MONGO_URI"])
mongo_db = mongo_client[os.environ.get("MONGO_DATABASE", "libflow")]
book_metadata = mongo_db["book_metadata"]

print("Connected to PostgreSQL and MongoDB.")

## 4. Load nested MongoDB documents
The `book_id` is the stable identifier shared with PostgreSQL `books.book_id`. MongoDB documents are intentionally not identical: each book can have a different number of editions, reviews, keywords and related topics.

In [ ]:
mongo_samples = [
    {
        "book_id": "BK001",
        "metadata": {
            "subjects": ["software engineering", "clean code", "programming"],
            "languages": ["English"],
            "keywords": ["refactoring", "readability", "testing"],
            "editions": [
                {"edition": 1, "year": 2008, "format": "hardcover"},
                {"edition": 2, "year": 2020, "format": "ebook"}
            ],
            "reviews": [
                {"reviewer": "student_demo_01", "rating": 5, "comment": "Useful examples."}
            ],
            "related_topics": {"course": ["Software Engineering"], "skills": ["refactoring"]}
        }
    },
    {
        "book_id": "BK002",
        "metadata": {
            "subjects": ["algorithms", "data structures"],
            "languages": ["English"],
            "keywords": ["graphs", "sorting"],
            "editions": [{"edition": 3, "year": 2009, "format": "hardcover"}],
            "reviews": [],
            "related_topics": {"course": ["Algorithms"], "skills": ["problem solving"]}
        }
    },
    {
        "book_id": "BK003",
        "metadata": {
            "subjects": ["Python", "programming"],
            "languages": ["English", "Japanese"],
            "keywords": ["data model", "iterators", "concurrency"],
            "editions": [
                {"edition": 1, "year": 2015, "format": "paperback"},
                {"edition": 2, "year": 2022, "format": "paperback"}
            ],
            "reviews": [{"reviewer": "student_demo_04", "rating": 5, "comment": "Detailed examples."}],
            "related_topics": {"course": ["Python Programming"], "skills": ["Python", "OOP"]}
        }
    }
]

for doc in mongo_samples:
    book_metadata.update_one(
        {"book_id": doc["book_id"]},
        {"$set": doc},
        upsert=True
    )

print("MongoDB sample metadata inserted/updated.")

## 5. MongoDB nested-document queries
Examples of querying nested arrays and nested fields:
- Find books whose subjects include `algorithms`.
- Find documents with an edition published in or after 2020.

In [ ]:
algorithm_books = list(book_metadata.find(
    {"metadata.subjects": "algorithms"},
    {"_id": 0, "book_id": 1, "metadata.subjects": 1}
))
print("Books about algorithms:", algorithm_books)

recent_edition_books = list(book_metadata.find(
    {"metadata.editions.year": {"$gte": 2020}},
    {"_id": 0, "book_id": 1, "metadata.editions": 1}
))
print("Books with a 2020-or-later edition:", recent_edition_books)

## 6. MongoDB → PostgreSQL integration
MongoDB keeps the complete flexible document. The Python bridge extracts only selected fields (subjects, languages and latest edition year) and synchronizes them into PostgreSQL `book_metadata_index`.

The shared `book_id` is an application-level integration key. PostgreSQL enforces its own foreign key from `book_metadata_index.book_id` to `books.book_id`; MongoDB itself does not participate in PostgreSQL foreign-key enforcement.

In [ ]:
def sync_mongo_metadata_to_postgres():
    docs = list(book_metadata.find({}, {"_id": 0, "book_id": 1, "metadata": 1}))
    rows = []
    for doc in docs:
        meta = doc.get("metadata", {})
        edition_years = [e.get("year") for e in meta.get("editions", []) if e.get("year")]
        latest_year = max(edition_years) if edition_years else None
        rows.append((
            doc["book_id"],
            meta.get("subjects", []),
            meta.get("languages", []),
            latest_year
        ))

    with pg_conn:
        with pg_conn.cursor() as cur:
            for row in rows:
                cur.execute(
                    '''
                    INSERT INTO book_metadata_index
                        (book_id, subjects, languages, latest_edition_year, synced_at)
                    VALUES (%s, %s, %s, %s, CURRENT_TIMESTAMP)
                    ON CONFLICT (book_id) DO UPDATE SET
                        subjects = EXCLUDED.subjects,
                        languages = EXCLUDED.languages,
                        latest_edition_year = EXCLUDED.latest_edition_year,
                        synced_at = CURRENT_TIMESTAMP
                    ''',
                    row
                )
    return len(rows)

print("Documents synchronized:", sync_mongo_metadata_to_postgres())

## 7. SQL CRUD examples

In [ ]:
# CREATE: add a demo student (change the ID/email if rerunning with existing data)
with pg_conn:
    with pg_conn.cursor() as cur:
        cur.execute(
            '''
            INSERT INTO students(student_id, full_name, email, department, semester, section)
            VALUES (%s, %s, %s, %s, %s, %s)
            ON CONFLICT (student_id) DO NOTHING
            ''',
            ("STU005", "Diya Menon", "diya@example.edu", "AIML", 3, "A")
        )

# READ
students_df = pd.read_sql(
    "SELECT student_id, full_name, department, semester FROM students ORDER BY student_id",
    pg_conn
)
display(students_df)

# UPDATE
with pg_conn:
    with pg_conn.cursor() as cur:
        cur.execute(
            "UPDATE students SET section = %s WHERE student_id = %s",
            ("B", "STU005")
        )

# DELETE example (only deletes the demo student if it has no issue records)
with pg_conn:
    with pg_conn.cursor() as cur:
        cur.execute("DELETE FROM students WHERE student_id = %s", ("STU005",))

## 8. SQL JOIN – issue details
This query combines three related tables. It demonstrates why relational keys are useful for operational reports.

In [ ]:
join_query = '''
SELECT i.issue_id, s.full_name, b.title, i.issue_date, i.due_date, i.status
FROM issues AS i
JOIN students AS s ON s.student_id = i.student_id
JOIN books AS b ON b.book_id = i.book_id
ORDER BY i.issue_id
'''
display(pd.read_sql(join_query, pg_conn))

## 9. SQL nested query – books never issued
A `NOT EXISTS` subquery identifies books that have no matching row in `issues`.

In [ ]:
nested_query = '''
SELECT b.book_id, b.title
FROM books AS b
WHERE NOT EXISTS (
    SELECT 1
    FROM issues AS i
    WHERE i.book_id = b.book_id
)
ORDER BY b.title
'''
display(pd.read_sql(nested_query, pg_conn))

## 10. Transaction – issue a book safely
The transaction locks the selected book row, checks availability, inserts the issue and commits. The database trigger decreases `available_copies`. If any step fails, the transaction rolls back.

Use a student ID and book ID that exist in your sample tables. The example is safe to rerun only after the previous active issue is returned or you choose another book.

In [ ]:
def issue_book(student_id, book_id, loan_days=14):
    try:
        with pg_conn:
            with pg_conn.cursor() as cur:
                cur.execute(
                    "SELECT title, available_copies FROM books WHERE book_id=%s FOR UPDATE",
                    (book_id,)
                )
                result = cur.fetchone()
                if result is None:
                    raise ValueError("Book ID does not exist.")
                title, available = result
                if available <= 0:
                    raise ValueError("No copies are currently available.")

                cur.execute(
                    '''
                    INSERT INTO issues
                        (student_id, book_id, issue_date, due_date, status,
                         title_snapshot, fine_per_day_snapshot)
                    VALUES (%s, %s, CURRENT_DATE, CURRENT_DATE + %s,
                            'issued', %s, 5.00)
                    RETURNING issue_id
                    ''',
                    (student_id, book_id, loan_days, title)
                )
                issue_id = cur.fetchone()[0]
        return issue_id
    except Exception:
        pg_conn.rollback()
        raise

# Example: uncomment and run once when STU003 and BK004 exist and a copy is available.
# new_issue_id = issue_book("STU003", "BK004")
# print("Created issue:", new_issue_id)

## 11. Return a book and assess the fine
The issue row is locked and updated in one transaction. The trigger restores the available copy and assesses the fine using the saved ₹5/day rate. `title_snapshot` and `fine_per_day_snapshot` preserve the terms applicable when the book was issued, even if the catalogue is later edited.

In [ ]:
def return_book(issue_id, returned_on=None):
    returned_on = returned_on or date.today()
    with pg_conn:
        with pg_conn.cursor() as cur:
            cur.execute(
                "SELECT status FROM issues WHERE issue_id=%s FOR UPDATE",
                (issue_id,)
            )
            row = cur.fetchone()
            if row is None:
                raise ValueError("Issue ID not found.")
            if row[0] == "returned":
                raise ValueError("This issue has already been returned.")
            cur.execute(
                '''
                UPDATE issues
                SET status='returned', return_date=%s
                WHERE issue_id=%s
                ''',
                (returned_on, issue_id)
            )

# Example only: replace with an active issue_id from your database.
# return_book(1)

## 12. Fine and payment report

In [ ]:
fine_report = '''
SELECT f.fine_id, s.full_name, b.title, f.amount, f.status
FROM fines f
JOIN issues i ON i.issue_id = f.issue_id
JOIN students s ON s.student_id = i.student_id
JOIN books b ON b.book_id = i.book_id
ORDER BY f.fine_id
'''
display(pd.read_sql(fine_report, pg_conn))

## 13. Hybrid design justification and limitations

**Why PostgreSQL?** Students, books, issues, fines and payments have predictable columns and strong relationships. Primary keys, foreign keys, CHECK constraints, transactions and JOINs protect important library operations.

**Why MongoDB?** Book metadata can vary by title: one book may have multiple editions, languages, reviews, keywords and nested course/skill mappings. A document can hold these optional nested structures without adding many nullable relational columns or creating several extra tables.

**Why not put everything in MongoDB?** Issue/return, stock counts, fines and payments require consistent multi-row updates and relational reporting. SQL transactions and constraints are a natural fit.

**Why not put everything in SQL?** SQL can store JSON/JSONB, so MongoDB is not technically mandatory. If metadata is small and stable, PostgreSQL alone is simpler and would be a valid design. MongoDB is justified here only if the application needs independently evolving, nested metadata and document-oriented queries.

**Cross-database limitation:** PostgreSQL cannot create a normal foreign key directly into MongoDB. The application uses the same immutable `book_id`, validates that the SQL book exists before synchronization, and logs/monitors synchronization in a production version. A failed sync can be retried; the SQL operational transaction must not depend on MongoDB being available.

**Snapshot rule:** `issues.title_snapshot` and `fine_per_day_snapshot` preserve the book title and fine policy at issue time. A later catalogue edit or fine-policy change must not rewrite historical issue records.

## 14. Verification queries

In [ ]:
# Inventory consistency
display(pd.read_sql(
    "SELECT book_id, title, total_copies, available_copies FROM books ORDER BY book_id",
    pg_conn
))

# Active overdue issues
overdue_query = '''
SELECT i.issue_id, s.full_name, b.title, i.due_date,
       (CURRENT_DATE - i.due_date) AS overdue_days
FROM issues i
JOIN students s ON s.student_id = i.student_id
JOIN books b ON b.book_id = i.book_id
WHERE i.status = 'issued' AND i.due_date < CURRENT_DATE
ORDER BY i.due_date
'''
display(pd.read_sql(overdue_query, pg_conn))

## 15. Conclusion
LibFlow uses PostgreSQL for reliable library transactions and MongoDB for flexible nested book metadata. Python connects both systems and synchronizes selected metadata through the stable `book_id`. The implementation demonstrates keys, constraints, CRUD, JOIN, nested query, triggers, transactions and a clear explanation of hybrid-database trade-offs.

## 16. Close connections

In [ ]:
pg_conn.close()
mongo_client.close()
print("Database connections closed.")